In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"

In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,intervention,174,0,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,intervention,174,0,0.0
2,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,2,intervention,174,0,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,intervention,174,0,0.0
4,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,3,intervention,174,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,ylls,cause,other_causes,other_causes,95_plus,severe,3,zero,54,0,0.0
539996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,zero,54,0,0.0
539997,ylls,cause,other_causes,other_causes,95_plus,severe,4,zero,54,0,0.0
539998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,zero,54,0,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  591306.980884
                                  2                  469119.806421
                                  3                  412121.233930
                                  4                  376933.536670
                                  5                  347363.884977
intervention  maternal_disorders  1                  591306.980884
                                  2                  469119.806421
                                  3                  412121.233930
                                  4                  376933.536670
                                  5                  347363.884977
zero          maternal_disorders  1                  597682.299097
                                  2                  474549.417177
                                  3                  416257.523958
                                  4                  380897.687313
            

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylds,cause,all_causes,all_causes,10_to_14,invalid,1,intervention,174,0,0.621377
1,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,intervention,174,0,0.000000
2,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,intervention,174,0,0.000000
3,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,intervention,174,0,0.000000
4,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,intervention,174,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
1889995,ylds,cause,pregnancy,parturition,95_plus,severe,5,zero,54,0,0.000000
1889996,ylds,cause,pregnancy,postpartum,95_plus,severe,5,zero,54,0,0.000000
1889997,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,zero,54,0,0.000000
1889998,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,zero,54,0,0.000000


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  117697.275511
                                  2                   93116.577964
                                  3                   75010.563497
                                  4                   62893.286069
                                  5                   45659.951633
              maternal_disorders  1                     164.936997
                                  2                     114.588556
                                  3                      92.849802
                                  4                      96.982363
                                  5                      98.453235
intervention  anemia              1                  117697.275511
                                  2                   93116.577964
                                  3                   75010.563497
                                  4                   62893.286069
            

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  117697.275511
                                  2                   93116.577964
                                  3                   75010.563497
                                  4                   62893.286069
                                  5                   45659.951633
              maternal_disorders  1                  591471.917880
                                  2                  469234.394977
                                  3                  412214.083731
                                  4                  377030.519033
                                  5                  347462.338212
intervention  anemia              1                  117697.275511
                                  2                   93116.577964
                                  3                   75010.563497
                                  4                   62893.286069
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert len(pd.read_parquet(ylds_path)) == 0

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    neonatal_ylls = pd.read_parquet(path).rename(
        columns={"maternal_scenario": "scenario"}
    )
else:
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,ylls,cause,stillborn,stillborn,0_to_6_months,Female,1,baseline,baseline,27,0,0.000000
1,ylls,cause,stillborn,stillborn,0_to_6_months,Female,2,baseline,baseline,27,0,0.000000
2,ylls,cause,stillborn,stillborn,0_to_6_months,Female,3,baseline,baseline,27,0,0.000000
3,ylls,cause,stillborn,stillborn,0_to_6_months,Female,4,baseline,baseline,27,0,0.000000
4,ylls,cause,stillborn,stillborn,0_to_6_months,Female,5,baseline,baseline,27,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
47995,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,103,0,6173.747288
47996,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,103,0,4583.326867
47997,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,103,0,4326.556829
47998,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,103,0,1874.647052


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.858785e+07
                      2                  1.536768e+07
                      3                  1.374935e+07
                      4                  1.289434e+07
                      5                  1.233837e+07
intervention  lbwsg   1                  1.858785e+07
                      2                  1.536768e+07
                      3                  1.374935e+07
                      4                  1.289434e+07
                      5                  1.233837e+07
zero          lbwsg   1                  1.864171e+07
                      2                  1.540172e+07
                      3                  1.377726e+07
                      4                  1.291693e+07
                      5                  1.235121e+07
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,1887.182086,zero
1,Female,0.0,0.019178,2,1500.591341,zero
2,Female,0.0,0.019178,3,1311.055992,zero
3,Female,0.0,0.019178,4,1113.656053,zero
4,Female,0.0,0.019178,5,794.175203,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,827.889366,intervention
746,Male,95.0,125.000000,2,844.741565,intervention
747,Male,95.0,125.000000,3,868.385175,intervention
748,Male,95.0,125.000000,4,894.404754,intervention


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  1.697623e+06
                      2                  1.647948e+06
                      3                  1.670624e+06
                      4                  1.576390e+06
                      5                  1.394833e+06
intervention  anemia  1                  1.697623e+06
                      2                  1.647948e+06
                      3                  1.670624e+06
                      4                  1.576390e+06
                      5                  1.394833e+06
zero          anemia  1                  1.833633e+06
                      2                  1.783349e+06
                      3                  1.792583e+06
                      4                  1.677701e+06
                      5                  1.446965e+06
Name: value, dtype: float64

In [18]:
scenarios[1]

'zero'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

-566098.3988143988

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  631771.028209
                      2                  510274.506138
                      3                  465500.810150
                      4                  393418.564420
                      5                  298642.960029
intervention  anemia  1                  631771.028209
                      2                  510274.506138
                      3                  465500.810150
                      4                  393418.564420
                      5                  298642.960029
zero          anemia  1                  675531.930427
                      2                  548538.037342
                      3                  497417.002078
                      4                  417322.321033
                      5                  309282.876152
Name: value, dtype: float64

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

-148484.29808647977

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  4.118092e+06
                      2                  3.703198e+06
                      3                  3.630130e+06
                      4                  3.328340e+06
                      5                  2.956617e+06
intervention  anemia  1                  4.118092e+06
                      2                  3.703198e+06
                      3                  3.630130e+06
                      4                  3.328340e+06
                      5                  2.956617e+06
zero          anemia  1                  4.471866e+06
                      2                  4.022986e+06
                      3                  3.906851e+06
                      4                  3.548791e+06
                      5                  3.065630e+06
Name: value, dtype: float64

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  397341.112053
                      2                  347011.089028
                      3                  308563.038948
                      4                  288866.787365
                      5                  239848.520376
baseline      ntd     1                  368777.931735
                      2                  325776.815913
                      3                  292471.014886
                      4                  275785.753509
                      5                  234875.149488
intervention  ntd     1                  208545.542613
                      2                  198697.663117
                      3                  190622.437118
                      4                  189344.581396
                      5                  196706.323192
Name: value, dtype: float64

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  4.235789e+06
                                  2                  3.796315e+06
                                  3                  3.705140e+06
                                  4                  3.391234e+06
                                  5                  3.002277e+06
              lbwsg               1                  1.858785e+07
                                  2                  1.536768e+07
                                  3                  1.374935e+07
                                  4                  1.289434e+07
                                  5                  1.233837e+07
              maternal_disorders  1                  5.914719e+05
                                  2                  4.692344e+05
                                  3                  4.122141e+05
                                  4                  3.770305e+05
                          

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)